In [0]:
-- =============================================================
-- GOLD LAYER — Business Models & KPIs
-- =============================================================


-- -------------------------------------------------------------
-- DIMENSION: dim_customer
-- One row per customer with enriched phone metadata
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.dim_customer AS
SELECT
  c.customer_id,
  c.customer_name,
  c.phone_number,
  c.country_code,
  c.country_name,
  c.line_type,
  c.carrier,
  c.plan_type,
  c.contract_type,
  c.payment_method,
  c.signup_date,
  c.tenure_months,
  c.is_churned,
  p.country_prefix,
  p.international_format,
  p.local_format,
  p.valid AS phone_valid
FROM etl.silver.customers c
LEFT JOIN etl.silver.phone_data p
  ON  c.carrier       = p.carrier
  AND c.country_code  = p.country_code
  AND c.line_type     = p.line_type;


-- -------------------------------------------------------------
-- FACT: fact_billing
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.fact_biling AS
SELECT
  b.usage_id,
  b.customer_id,
  b.phone_number,
  b.billing_month,
  b.minutes_used,
  b.data_used_gb,
  b.sms_sent,
  b.monthly_charge_usd,
  c.customer_name,
  c.country_code,
  c.country_name,
  c.carrier,
  c.plan_type,
  c.contract_type,
  c.line_type,
  c.is_churned,
  c.tenure_months
FROM etl.silver.biling b
LEFT JOIN etl.silver.customers c
  ON b.customer_id = c.customer_id;


-- -------------------------------------------------------------
-- KPI 1: Monthly revenue by carrier
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.kpi_revenue_by_carrier AS
SELECT
  billing_month,
  carrier,
  COUNT(DISTINCT customer_id)       AS active_customers,
  ROUND(SUM(monthly_charge_usd), 2) AS total_revenue_usd,
  ROUND(AVG(monthly_charge_usd), 2) AS avg_revenue_usd         
FROM etl.gold.fact_biling
GROUP BY billing_month, carrier
ORDER BY billing_month DESC, total_revenue_usd DESC;


-- -------------------------------------------------------------
-- KPI 2: Churn summary by plan type
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.kpi_churn_by_plan AS
SELECT
  plan_type,
  contract_type,
  COUNT(*) AS total_customers,
  SUM(CASE WHEN is_churned = TRUE THEN 1 ELSE 0 END) AS churned_customers,
  ROUND( 100.0 * SUM(CASE WHEN is_churned = TRUE THEN 1 ELSE 0 END) 
  / COUNT(*),2) AS churn_rate_pct
FROM etl.gold.dim_customer
GROUP BY plan_type, contract_type
ORDER BY churn_rate_pct DESC;


-- -------------------------------------------------------------
-- KPI 3: Usage summary by country
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.kpi_usage_by_country AS
SELECT
  country_name,
  country_code,
  billing_month,
  COUNT(DISTINCT customer_id)         AS subscribers,
  ROUND(SUM(minutes_used), 0)         AS total_minutes,
  ROUND(SUM(data_used_gb), 2)         AS total_data_gb,
  SUM(sms_sent)                       AS total_sms,
  ROUND(SUM(monthly_charge_usd), 2)   AS total_revenue_usd
FROM etl.gold.fact_biling
GROUP BY country_name, country_code, billing_month
ORDER BY billing_month DESC, total_revenue_usd DESC;


-- -------------------------------------------------------------
-- KPI 4: Customer lifetime value (CLV) estimate
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE etl.gold.kpi_customer_clv AS
SELECT
  customer_id,
  customer_name,
  carrier,
  plan_type,
  tenure_months,
  is_churned,
  ROUND(SUM(monthly_charge_usd), 2) AS total_spend_usd,
  ROUND(AVG(monthly_charge_usd), 2) AS avg_monthly_spend_usd,
  ROUND(AVG(monthly_charge_usd) * tenure_months,2) AS estimated_clv_usd
FROM etl.gold.fact_biling
GROUP BY customer_id, customer_name, carrier, plan_type, tenure_months, is_churned
ORDER BY estimated_clv_usd DESC;